<a href="https://colab.research.google.com/github/Musharrafmrm/hybrid-ecommerce-issue-detection/blob/main/notebooks/09_Hybrid_Model_Training_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================================
# 09_Hybrid_Model_Training_Final.ipynb
# COMPLETE HYBRID MODEL (Self-Contained)
# =============================================================================

import pandas as pd
import re
import os
import zipfile
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
import joblib
import torch
from transformers import BertTokenizer, BertModel
from tqdm import tqdm

nltk.download('stopwords', quiet=True)

print("🔄 Starting Full Hybrid Model Pipeline...\n")

os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)

# ====================== 1. Load or Download Data ======================
if os.path.exists('data/cleaned_dataset.csv'):
    df = pd.read_csv('data/cleaned_dataset.csv')
    print("✅ Loaded cleaned dataset")
else:
    print("Downloading and preparing Amazon dataset...")
    if not os.path.exists('data/Reviews.csv'):
        !kaggle datasets download -d snap/amazon-fine-food-reviews -f Reviews.csv --path data/ --unzip
        if os.path.exists('data/Reviews.csv.zip'):
            with zipfile.ZipFile('data/Reviews.csv.zip', 'r') as z:
                z.extractall('data/')
    df = pd.read_csv('data/Reviews.csv')
    df = df[['Score', 'Summary', 'Text']].copy()

    def clean_text(text):
        if not isinstance(text, str):
            return ""
        text = text.lower()
        text = re.sub(r'[^a-zA-Z\s]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    df['cleaned_text'] = df['Text'].apply(clean_text)
    df = df[df['cleaned_text'].str.len() > 15]
    df.to_csv('data/cleaned_dataset.csv', index=False)
    print("✅ Data cleaned and saved")

print(f"Total reviews: {len(df):,}")

# Use 20,000 reviews for reasonable training time
df = df.sample(n=20000, random_state=42).reset_index(drop=True)

# Create label (0 = Issue, 1 = No Issue)
df['label'] = df['Score'].apply(lambda x: 0 if x <= 3 else 1)

# ====================== 2. LDA (Aspect Discovery) ======================
print("\nTraining LDA...")
stop_words = stopwords.words('english')
vectorizer = CountVectorizer(max_df=0.95, min_df=2, stop_words=stop_words, max_features=5000)
X_text = vectorizer.fit_transform(df['cleaned_text'])

lda = LatentDirichletAllocation(n_components=5, random_state=42, max_iter=10)
lda.fit(X_text)

joblib.dump(lda, 'models/lda_model.pkl')
joblib.dump(vectorizer, 'models/vectorizer.pkl')
print("✅ LDA completed and saved")

# Add LDA features
topic_dist = lda.transform(X_text)
for i in range(5):
    df[f'topic_{i+1}'] = topic_dist[:, i]

# ====================== 3. BERT Embeddings ======================
print("\nGenerating BERT embeddings...")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device.upper()}")

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')
model = model.to(device)
model.eval()

def get_bert_embedding(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()

bert_embeddings = []
for text in tqdm(df['cleaned_text'], desc="BERT"):
    emb = get_bert_embedding(text)
    bert_embeddings.append(emb)

bert_features = np.array(bert_embeddings)

# Combine features
lda_features = df[[col for col in df.columns if col.startswith('topic_')]].values
X = np.hstack((lda_features, bert_features))
y = df['label'].values

print(f"Final feature shape: {X.shape}")

# ====================== 4. Train Hybrid Model ======================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\nTraining Random Forest...")
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))

print("\nTraining SVM...")
svm = SVC(kernel='rbf', C=1.0, random_state=42)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)
print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))

# Save models
joblib.dump(rf, 'models/random_forest_hybrid.pkl')
joblib.dump(svm, 'models/svm_hybrid.pkl')

print("\n🎉 HYBRID MODEL TRAINING COMPLETED SUCCESSFULLY!")

🔄 Starting Full Hybrid Model Pipeline...

Dataset URL: https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews
License(s): CC0-1.0
100% 115M/115M [00:01<00:00, 109MB/s]

✅ Data cleaned and saved
Total reviews: 568,452

Training LDA...
✅ LDA completed and saved

Generating BERT embeddings...
Using device: CPU


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
BERT: 100%|██████████| 20000/20000 [1:54:04<00:00,  2.92it/s]


NameError: name 'np' is not defined

In [ ]:
import numpy as np
bert_features = np.array(bert_embeddings)
print("✅ BERT features converted successfully!")
print(f"Shape: {bert_features.shape}")

NameError: name 'bert_embeddings' is not defined